In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt 
from data_processing.processing.figure_of_merit import fit_fom, FOM, gaussian
from data_processing.reporting.plotting import plot_fom
from scipy.optimize import curve_fit
from scipy.io import loadmat

In [ ]:
ROOT = Path("C:/Users/Neutron Computer/Desktop/MATLAB Results/Maddy's experiment Oct 14, 22/echem_exp")
AMPLITUDES_PATH = ROOT / "amplitudes.csv"
PSD_PATH =  ROOT / "psd.csv"
SIGNAL_PATH = ROOT / "signalArray.mat"

In [ ]:
psd = pd.read_csv(PSD_PATH, header=None)
amplitudes = pd.read_csv(AMPLITUDES_PATH, header=None)

psd_report = pd.concat([psd, amplitudes], axis=1)
psd_report.columns = ["tail / total", "amplitude"]
psd_report.head()

In [ ]:
arr = loadmat(SIGNAL_PATH)

In [ ]:
signal = pd.DataFrame(arr["signalArray"])
signal.head()

In [ ]:
def find_base_idx(signal):
    peak_idx = signal.idxmax()
    return peak_idx - signal[peak_idx-25:peak_idx].argmin()

In [ ]:
base_idxs = signal.apply(find_base_idx)

In [ ]:
base_idxs.head()

In [ ]:
def integrate_tenth(signal, base_idxs):
    print(f"Processing # {signal.name}")
    res = []
    sig = signal
    long_gate = sig[base_idxs[sig.name]:]
    splice_len = int(np.floor(len(long_gate) / 10))
    
    curr_idx = base_idxs[sig.name]
    for i in range(0, 10):    
        if i == 9:
            res.append(
                sig[curr_idx:].sum()
            )
        else:
            res.append(
                sig[curr_idx: curr_idx + splice_len].sum()
            )

        curr_idx = curr_idx + splice_len
    
    return res
result = signal.apply(lambda x: integrate_tenth(x, base_idxs))

In [ ]:
result

In [ ]:
plt.bar(range(0,10), result[10])

In [ ]:
from sklearn import neighbors
from sklearn.model_selection import train_test_split

In [ ]:
psd_train, psd_test = train_test_split(psd, test_size=0.2)
features_train, features_test = result.iloc[:,psd_train.index], result.iloc[:,psd_test.index]

In [ ]:
n_neighbors = 5
knn = neighbors.KNeighborsRegressor(n_neighbors, weights="uniform")

In [ ]:
model = knn.fit(features_train.T, psd_train)

In [ ]:
test_results = model.predict(result.T)

In [ ]:
plt.scatter(
    amplitudes.iloc[psd_test.index],
    psd.iloc[psd_test.index],
    alpha = 0.3,
)

plt.scatter(
    amplitudes,
    test_results,
)



In [ ]:
counts, bins = np.histogram(test_results, 100)
plt.plot(bins[:-1], counts)
# plt.yscale("log")